In [ ]:
#REMEMBER TO SPECIFY GPU OR TPU BEFORE STARTING


In [ ]:
#LOAD VARIOUS LIBRARIES

!pip install -U sentence-transformers
import pandas, numpy
import pickle

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
#https://huggingface.co/sentence-transformers/all-mpnet-base-v2'

#I used best performing model from here: https://www.sbert.net/docs/pretrained_models.html

In [ ]:
#MOUNT DRIVE

from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
#FUNCTION FOR GETING REASON VECTORS FOR EACH CHUNK OF DATA

#function takes chunk's dataframe as input along with the chunknumber and subreddit name
def process_chunk(df_chunk,chunknum,subreddit):

  #empty dictionary
  reasondict = {}

  #iterate over reasons types
  for col in 'benefits1_1,benefits1_2,benefits1_3,costs1_1,costs1_2,costs1_3,benefits2_1,benefits2_2,benefits2_3,costs2_1,costs2_2,costs2_3'.split(','):

    #iterate over individual reasons
    for r in list(df_chunk[col]):

      #make sure it is string
      if type(r) == str and r not in reasondict:

        #get reason vector and put in dictionary
        rvec = model.encode([r])
        reasondict[r] = rvec[0]

  #save dictionary
  with open('/content/drive/MyDrive/Colab Data - Advice/output - reasondict - ' + subreddit + ' - '  +str(chunknum) + '.pkl', 'wb') as handle:
      pickle.dump(reasondict, handle, protocol=pickle.HIGHEST_PROTOCOL)



In [ ]:
#CALL THE ABOVE FUNTION FOR VARIOUS SUBREDDITS

#specify subreddit
subreddit = 'Advice'

#file path on google drive for the original data (e.g. "data - Advice.csv")
directory = '...'
file_path = directory + '/data - ' + subreddit + '.csv'

#specify chunksize for vectorizing
chunk_size = 100

#read file and iterate over chunks
reader = pandas.read_csv(file_path, chunksize=chunk_size)
chunknum = 0
for chunk in reader:
    process_chunk(chunk,chunknum,subreddit)
    chunknum = chunknum + 1

In [ ]:
#GET VECTORS FOR OUR OWN ATTRIBUTES

#load attributes
df_at = pandas.read_csv('attributes.csv', encoding='cp1252')

#create empty dictionary
attributesdict = {}

#iterate over data frame
for i, row in df_at.iterrows():

  #specify attribute
  attribute = row['Name']

  #specify direction (pro or con)
  direction = row['Direction']

  #generate name
  name = attribute+"_"+direction

  #extract component sentences
  sentences = row['Sentences'].replace('.','').split('; ')

  #vectorize and average component sentences
  vecs = model.encode(sentences)
  vec_ave = numpy.mean(vecs, axis = 0)

  #put in dictionary
  attributesdict[name] = list(vec_ave)

#save dictionary
with open('output - attributesdict.pkl', 'wb') as handle:
    pickle.dump(attributesdict, handle, protocol=pickle.HIGHEST_PROTOCOL)
